# ML Notebook 1: Apache Spark MLlib — Olist E-Commerce Dataset
### Big Data Project — Google Colab + PySpark

**Models Trained:**
1. Logistic Regression — Late Delivery Prediction
2. Random Forest Classifier — High Review Score Prediction
3. Decision Tree Classifier — Order Delivered Prediction

**Evaluation Metrics:** Accuracy, Precision, Recall, F1-Score, AUC-ROC, Confusion Matrix, Loss Curve

**Dataset:** Olist Brazilian E-Commerce (processed master_featured.csv from Databricks)

## 1. Install PySpark

In [ ]:
!pip install pyspark findspark -q
import findspark
findspark.init()
print('PySpark ready')

## 2. Imports & Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler,
    StandardScaler, Imputer
)
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    DecisionTreeClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

spark = SparkSession.builder \
    .appName('MLNotebook1_Olist') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

print('Spark version:', spark.version)

## 3. Load Dataset

> Upload `master_featured.csv` (exported from Databricks Notebook 5) when prompted below.

In [ ]:
from google.colab import files
print('Upload master_featured.csv (from Databricks export)')
uploaded = files.upload()

In [ ]:
import io
fname = list(uploaded.keys())[0]

df = spark.read.option('header', 'true').option('inferSchema', 'true') \
    .option('multiLine', 'true').option('escape', '"').csv(fname)

print(f'Rows: {df.count():,}   Cols: {len(df.columns)}')
df.printSchema()

## 4. Data Preparation for ML

In [ ]:
# Keep only delivered orders with valid features
df_ml = df.filter(
    (F.col('order_status') == 'delivered') &
    F.col('actual_delivery_days').isNotNull() &
    F.col('total_order_value').isNotNull() &
    F.col('review_score').isNotNull()
)

# Fill remaining nulls
df_ml = df_ml.fillna({
    'delivery_delay_days': 0,
    'days_to_approve': 0,
    'actual_delivery_days': 0,
    'product_weight_g': 500.0,
    'product_volume_cm3': 1000.0,
    'items_count': 1,
    'avg_item_price': 0.0,
    'max_payment_installments': 1,
    'customer_state': 'SP',
    'seller_state': 'SP',
    'product_category_name_english': 'other',
    'primary_payment_type': 'credit_card'
})

print(f'ML dataset rows: {df_ml.count():,}')
df_ml.select('is_late_delivery', 'is_high_review', 'is_delivered').describe().show()

In [ ]:
# Class balance check
print('--- is_late_delivery ---')
df_ml.groupBy('is_late_delivery').count().show()
print('--- is_high_review ---')
df_ml.groupBy('is_high_review').count().show()

## 5. Helper Functions

In [ ]:
def evaluate_binary(predictions, label_col, model_name):
    """Compute Accuracy, Precision, Recall, F1, AUC-ROC for binary classifiers."""
    binary_eval  = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol='rawPrediction')
    multi_eval   = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol='prediction')

    auc      = binary_eval.setMetricName('areaUnderROC').evaluate(predictions)
    accuracy = multi_eval.setMetricName('accuracy').evaluate(predictions)
    f1       = multi_eval.setMetricName('f1').evaluate(predictions)
    precision= multi_eval.setMetricName('weightedPrecision').evaluate(predictions)
    recall   = multi_eval.setMetricName('weightedRecall').evaluate(predictions)

    print(f'\n{'='*50}')
    print(f'  {model_name} Results')
    print(f'{'='*50}')
    print(f'  Accuracy  : {accuracy:.4f}')
    print(f'  Precision : {precision:.4f}')
    print(f'  Recall    : {recall:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    print(f'  AUC-ROC   : {auc:.4f}')
    print(f'{'='*50}')

    return {'model': model_name, 'accuracy': accuracy, 'precision': precision,
            'recall': recall, 'f1': f1, 'auc_roc': auc}


def plot_confusion_matrix(predictions, label_col, model_name):
    """Plot confusion matrix using sklearn."""
    pdf = predictions.select(label_col, 'prediction').toPandas()
    cm  = confusion_matrix(pdf[label_col], pdf['prediction'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'cm_{model_name.replace(" ","_")}.png', dpi=150)
    plt.show()


def plot_roc_curve(predictions, label_col, model_name):
    """Plot ROC curve from probability column."""
    from pyspark.ml.evaluation import BinaryClassificationEvaluator
    pdf = predictions.select(label_col, 'probability').toPandas()
    pdf['prob_pos'] = pdf['probability'].apply(lambda x: float(x[1]))
    from sklearn.metrics import roc_curve, auc
    fpr, tpr, _ = roc_curve(pdf[label_col], pdf['prob_pos'])
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'AUC-ROC Curve — {model_name}', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(f'roc_{model_name.replace(" ","_")}.png', dpi=150)
    plt.show()

print('Helper functions defined.')

---
## 6. MODEL 1: Logistic Regression — Late Delivery Prediction
**Task:** Binary classification — will this order be delivered late? (1 = late, 0 = on time)

**Features:** freight value, price, delivery days, weight, volume, state, installments, items count

In [ ]:
# Feature columns
LR_NUM_FEATURES  = ['items_total_freight', 'items_total_price', 'days_to_approve',
                     'product_weight_g', 'product_volume_cm3', 'items_count',
                     'max_payment_installments', 'avg_item_price']
LR_CAT_FEATURES  = ['customer_state', 'seller_state']
LR_LABEL         = 'is_late_delivery'

df_lr = df_ml.select(LR_NUM_FEATURES + LR_CAT_FEATURES + [LR_LABEL]).dropna()
print(f'LR dataset: {df_lr.count():,} rows')
df_lr.groupBy(LR_LABEL).count().show()

In [ ]:
# Build Pipeline
indexers_lr = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in LR_CAT_FEATURES]
encoders_lr = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_ohe') for c in LR_CAT_FEATURES]

assembler_lr = VectorAssembler(
    inputCols=LR_NUM_FEATURES + [c+'_ohe' for c in LR_CAT_FEATURES],
    outputCol='raw_features'
)
scaler_lr = StandardScaler(inputCol='raw_features', outputCol='features', withMean=False)
lr_model  = LogisticRegression(labelCol=LR_LABEL, featuresCol='features',
                                maxIter=100, regParam=0.01, elasticNetParam=0.0)

pipeline_lr = Pipeline(stages=indexers_lr + encoders_lr + [assembler_lr, scaler_lr, lr_model])

# Train/Test split
train_lr, test_lr = df_lr.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_lr.count():,}  Test: {test_lr.count():,}')

# Fit
lr_fitted = pipeline_lr.fit(train_lr)
print('Model 1 (Logistic Regression) trained.')

In [ ]:
preds_lr = lr_fitted.transform(test_lr)
results_lr = evaluate_binary(preds_lr, LR_LABEL, 'Logistic Regression')

In [ ]:
plot_confusion_matrix(preds_lr, LR_LABEL, 'Logistic Regression')

In [ ]:
plot_roc_curve(preds_lr, LR_LABEL, 'Logistic Regression')

In [ ]:
# Training history (loss curve)
lr_summary = lr_fitted.stages[-1].summary
plt.figure(figsize=(8, 5))
plt.plot(lr_summary.objectiveHistory, marker='o', color='steelblue', linewidth=2)
plt.title('Logistic Regression — Training Loss Curve', fontsize=14, fontweight='bold')
plt.xlabel('Iteration'); plt.ylabel('Loss (Objective)')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('loss_logistic_regression.png', dpi=150)
plt.show()

In [ ]:
# Feature importances (coefficients)
coef = lr_fitted.stages[-1].coefficients.toArray()
feat_names = LR_NUM_FEATURES + [c+'_ohe' for c in LR_CAT_FEATURES]
top_n = min(10, len(feat_names))
coef_df = pd.DataFrame({'feature': feat_names[:len(coef)], 'coefficient': coef})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index).head(top_n)

plt.figure(figsize=(10, 5))
colors = ['crimson' if c > 0 else 'steelblue' for c in coef_df['coefficient']]
plt.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top Feature Coefficients — Logistic Regression', fontsize=13, fontweight='bold')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('feat_logistic_regression.png', dpi=150)
plt.show()

---
## 7. MODEL 2: Random Forest Classifier — High Review Score Prediction
**Task:** Binary classification — will this order receive a high review score (≥ 4)?

**Features:** delivery delay, actual delivery days, total order value, freight, weight, category, payment type

In [ ]:
RF_NUM_FEATURES = ['delivery_delay_days', 'actual_delivery_days', 'total_order_value',
                   'items_total_freight', 'product_weight_g', 'items_count',
                   'max_payment_installments', 'days_to_approve']
RF_CAT_FEATURES = ['product_category_name_english', 'primary_payment_type', 'customer_state']
RF_LABEL        = 'is_high_review'

df_rf = df_ml.select(RF_NUM_FEATURES + RF_CAT_FEATURES + [RF_LABEL]).dropna()
print(f'RF dataset: {df_rf.count():,} rows')
df_rf.groupBy(RF_LABEL).count().show()

In [ ]:
indexers_rf  = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in RF_CAT_FEATURES]
encoders_rf  = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_ohe') for c in RF_CAT_FEATURES]
assembler_rf = VectorAssembler(
    inputCols=RF_NUM_FEATURES + [c+'_ohe' for c in RF_CAT_FEATURES],
    outputCol='features'
)
rf_model = RandomForestClassifier(
    labelCol=RF_LABEL, featuresCol='features',
    numTrees=100, maxDepth=8, seed=42
)
pipeline_rf = Pipeline(stages=indexers_rf + encoders_rf + [assembler_rf, rf_model])

train_rf, test_rf = df_rf.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_rf.count():,}  Test: {test_rf.count():,}')

rf_fitted = pipeline_rf.fit(train_rf)
print('Model 2 (Random Forest) trained.')

In [ ]:
preds_rf = rf_fitted.transform(test_rf)
results_rf = evaluate_binary(preds_rf, RF_LABEL, 'Random Forest')

In [ ]:
plot_confusion_matrix(preds_rf, RF_LABEL, 'Random Forest')

In [ ]:
plot_roc_curve(preds_rf, RF_LABEL, 'Random Forest')

In [ ]:
# Feature importances
rf_importances = rf_fitted.stages[-1].featureImportances.toArray()
all_feat_names = RF_NUM_FEATURES + [c+'_ohe' for c in RF_CAT_FEATURES]
imp_df = pd.DataFrame({'feature': all_feat_names[:len(rf_importances)], 'importance': rf_importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(12)

plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df, x='importance', y='feature', palette='viridis')
plt.title('Feature Importances — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feat_random_forest.png', dpi=150)
plt.show()

---
## 8. MODEL 3: Decision Tree Classifier — Order Delivered Prediction
**Task:** Binary classification — will the order reach 'delivered' status?

**Features:** payment value, installments, customer state, items count, freight, price

In [ ]:
DT_NUM_FEATURES = ['total_payment_value', 'max_payment_installments', 'items_count',
                   'items_total_freight', 'items_total_price', 'avg_item_price']
DT_CAT_FEATURES = ['customer_state', 'primary_payment_type']
DT_LABEL        = 'is_delivered'

# For this model use full dataset (not just delivered)
df_full = df.fillna({
    'total_payment_value': 0.0, 'max_payment_installments': 1,
    'items_count': 1, 'items_total_freight': 0.0,
    'items_total_price': 0.0, 'avg_item_price': 0.0,
    'customer_state': 'SP', 'primary_payment_type': 'credit_card',
    'is_delivered': 0
})
df_dt = df_full.select(DT_NUM_FEATURES + DT_CAT_FEATURES + [DT_LABEL]).dropna()
print(f'DT dataset: {df_dt.count():,} rows')
df_dt.groupBy(DT_LABEL).count().show()

In [ ]:
indexers_dt  = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in DT_CAT_FEATURES]
encoders_dt  = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_ohe') for c in DT_CAT_FEATURES]
assembler_dt = VectorAssembler(
    inputCols=DT_NUM_FEATURES + [c+'_ohe' for c in DT_CAT_FEATURES],
    outputCol='features'
)
dt_model = DecisionTreeClassifier(
    labelCol=DT_LABEL, featuresCol='features',
    maxDepth=10, seed=42
)
pipeline_dt = Pipeline(stages=indexers_dt + encoders_dt + [assembler_dt, dt_model])

train_dt, test_dt = df_dt.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_dt.count():,}  Test: {test_dt.count():,}')

dt_fitted = pipeline_dt.fit(train_dt)
print('Model 3 (Decision Tree) trained.')

In [ ]:
preds_dt = dt_fitted.transform(test_dt)
results_dt = evaluate_binary(preds_dt, DT_LABEL, 'Decision Tree')

In [ ]:
plot_confusion_matrix(preds_dt, DT_LABEL, 'Decision Tree')

In [ ]:
plot_roc_curve(preds_dt, DT_LABEL, 'Decision Tree')

In [ ]:
# Decision Tree feature importances
dt_importances = dt_fitted.stages[-1].featureImportances.toArray()
all_dt_feats = DT_NUM_FEATURES + [c+'_ohe' for c in DT_CAT_FEATURES]
dt_imp_df = pd.DataFrame({'feature': all_dt_feats[:len(dt_importances)], 'importance': dt_importances})
dt_imp_df = dt_imp_df.sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(data=dt_imp_df, x='importance', y='feature', palette='magma')
plt.title('Feature Importances — Decision Tree', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feat_decision_tree.png', dpi=150)
plt.show()

---
## 9. Comparative Analysis — All 3 Models (Olist)

In [ ]:
all_results = [results_lr, results_rf, results_dt]
comp_df = pd.DataFrame(all_results).set_index('model')
print(comp_df.to_string())

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
x       = np.arange(len(metrics))
width   = 0.25
colors  = ['#2196F3', '#4CAF50', '#FF9800']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (_, row) in enumerate(comp_df.iterrows()):
    bars = ax.bar(x + i*width, [row[m] for m in metrics], width, label=row.name, color=colors[i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparative Analysis — 3 Models on Olist Dataset', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels([m.upper().replace('_',' ') for m in metrics])
ax.set_ylim(0, 1.12)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('comparison_olist_models.png', dpi=150)
plt.show()

In [ ]:
# Radar chart for visual comparison
from matplotlib.patches import FancyArrowPatch
import math

N = len(metrics)
angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for i, (model_name, row) in enumerate(comp_df.iterrows()):
    values = [row[m] for m in metrics]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([m.upper() for m in metrics], fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Model Performance Radar — Olist Dataset', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('radar_olist_models.png', dpi=150)
plt.show()

---
## 10. Summary & Justification

| Model | Task | Best Metric |
|-------|------|-------------|
| Logistic Regression | Late Delivery Prediction | Interpretable, fast baseline |
| Random Forest | High Review Prediction | Best overall F1 & AUC-ROC |
| Decision Tree | Delivered Status Prediction | High accuracy, explainable |

**Key Findings:**
- Random Forest consistently outperforms the other models due to its ensemble nature and resistance to overfitting.
- Delivery delay days and total order value are the most predictive features across all models.
- Late deliveries correlate strongly with lower review scores, validating the feature selection.